## Regresión Lineal — Predicción de Precios de Autos Deportivos - Etapa de Limpieza de los datos

### Objetivo

El objetivo de este proyecto es preparar un dataset de autos depoRegresión Lineal — Predicción de Precios de Autos Deportivosrtivos para desarrollar posteriormente un modelo de regresión lineal capaz de predecir el precio de un vehículo a partir de sus características.

El dataset contiene información sobre marca, modelo, año, tamaño del motor, potencia, torque, tiempo de aceleración de 0 a 60 MPH y precio.

| Columna                   | Tipo conceptual    | Posible tratamiento                          |
| ------------------------- | ------------------ | -------------------------------------------- |
| `Car Make`                | Categórica nominal | Revisar tipo y limpiar                         |
| `Car Model`               | Categórica nominal | Analizar cardinalidad |
| `Year`                    | Numérica           | Mantener                                     |
| `Engine Size (L)`         | Numérica*          | Revisar tipo y limpiar                       |
| `Horsepower`              | Numérica*          | Revisar tipo y limpiar                       |
| `Torque (lb-ft)`          | Numérica*          | Revisar tipo y limpiar                       |
| `0-60 MPH Time (seconds)` | Numérica           | Mantener                                     |
| `Price (in USD)`          | Numérica           | 🎯 **Target**                                |


In [2]:
#importando librerias

import os
import pandas as pd 
import matplotlib.pyplot as plt 


In [3]:
#cargando el dataset
path_dataset = '../datasets/Sport-car-price.csv'
path_notebook = '../notebooks/Sport-car-price.ipynb'

df = pd.read_csv(path_dataset)

In [4]:
#Exploracion inicial del dataset

print("Dimensiones del dataset:", df.shape)
print("Columnas del datset:")
print(list(df.columns))
print("\nTipo de datos del dataset:")
print(df.dtypes)
print("\nPrimeras 5 filas del dataset:")
print(df.head())

Dimensiones del dataset: (1007, 8)
Columnas del datset:
['Car Make', 'Car Model', 'Year', 'Engine Size (L)', 'Horsepower', 'Torque (lb-ft)', '0-60 MPH Time (seconds)', 'Price (in USD)']

Tipo de datos del dataset:
Car Make                     str
Car Model                    str
Year                       int64
Engine Size (L)              str
Horsepower                   str
Torque (lb-ft)               str
0-60 MPH Time (seconds)      str
Price (in USD)               str
dtype: object

Primeras 5 filas del dataset:
      Car Make Car Model  Year Engine Size (L) Horsepower Torque (lb-ft)  \
0      Porsche       911  2022               3        379            331   
1  Lamborghini   Huracan  2021             5.2        630            443   
2      Ferrari   488 GTB  2022             3.9        661            561   
3         Audi        R8  2022             5.2        562            406   
4      McLaren      720S  2021               4        710            568   

  0-60 MPH Time (sec

In [5]:

#Busqueda de datos ratos entre las columnas de texto que deberian ser numericas
columnas_numericas_texto = ['Engine Size (L)', 'Horsepower', 'Torque (lb-ft)', '0-60 MPH Time (seconds)', 'Price (in USD)']

for columna in columnas_numericas_texto:
    print(f"\n--- {columna} ---")
    print(df[columna].unique()[:20])


--- Engine Size (L) ---
<StringArray>
[             '3',            '5.2',            '3.9',              '4',
            '4.4',            '6.2',            '3.8',              '8',
              '5',            '3.5',            '4.7',              '2',
            '2.9',              '6',       'Electric',            '6.5',
            '3.7', 'Electric Motor',            '2.5', '1.5 + Electric']
Length: 20, dtype: str

--- Horsepower ---
<StringArray>
[ '379',  '630',  '661',  '562',  '710',  '617',  '523',  '490',  '760',
  '600', '1500',  '717',  '296', '1280',  '471',  '416',  '454',  '300',
  '505',  '320']
Length: 20, dtype: str

--- Torque (lb-ft) ---
<StringArray>
[ '331',  '443',  '561',  '406',  '568',  '553',  '494',  '465',  '625',
  '481',  '516', '1180',  '656',  '295', '1015',  '398',  '317',  '384',
  '280',  '243']
Length: 20, dtype: str

--- 0-60 MPH Time (seconds) ---
<StringArray>
[   '4',  '2.8',    '3',  '3.2',  '2.7',  '3.1',  '3.8',  '3.5',  '2.5',
  '2.4', 

### Exploración inicial - Hallazgos

Durante la inspección inicial se identificó que varias variables conceptualmente numéricas fueron importadas como texto (`str`). Esto indica la posible presencia de caracteres, formatos especiales o valores no numéricos.

Por esta razón, estas variables serán investigadas individualmente antes de realizar cualquier transformación.

# Trabajando con la columna **Engine Size (L)**

In [6]:

print(df['Engine Size (L)'].value_counts().to_string())

Engine Size (L)
4                       219
6.2                     113
3                        85
3.5                      79
5                        68
6.5                      46
3.8                      38
Electric                 36
3.7                      35
2                        34
3.9                      30
2.9                      30
5.2                      29
6                        28
2.5                      25
8                        23
4.7                      23
4.4                      11
6.8                       6
1.7                       4
Electric Motor            3
8.4                       3
6.6                       3
1.8                       3
1.5                       2
Hybrid                    2
1.5 + Electric            1
7                         1
3.3                       1
-                         1
6.7                       1
Electric (tri-motor)      1
5.5                       1
Electric (93 kWh)         1
Electric (100 kWh)        1
Hybr

In [7]:
print(
    df[df['Engine Size (L)'].isin(['0', '-'])][
        ['Car Make', 'Car Model', 'Year', 'Engine Size (L)']
    ]
)

    Car Make Car Model  Year Engine Size (L)
335    Tesla  Roadster  2022               -
885    Tesla  Roadster  2022               0


In [8]:
def clasificar_motor(row):
    engine = row['Engine Size (L)']
    make = row['Car Make']
    model = row['Car Model']

    # Valor realmente faltante
    if pd.isna(engine):
        return 'Unknown'

    engine = str(engine)

    # Híbridos
    if 'Hybrid' in engine or '+' in engine:
        return 'Hybrid'

    # Eléctricos
    elif 'Electric' in engine:
        return 'Electric'

    # Tesla Roadster identificado durante la exploración
    elif make == 'Tesla' and model == 'Roadster' and engine in ['-', '0']:
        return 'Electric'

    # Valores numéricos convencionales
    else:
        return 'Combustion'


df['Engine Type'] = df.apply(clasificar_motor, axis=1)

print(df['Engine Type'].value_counts())

Engine Type
Combustion    947
Electric       45
Unknown        10
Hybrid          5
Name: count, dtype: int64


In [9]:
#Crear columna numerica para el tamaño del motor donde los valores invalidos se convierten en NaN
df['Engine Size Numeric'] = pd.to_numeric(
    df['Engine Size (L)'],
    errors='coerce'
)

# Los vehiculos electricos no tienen cilindrada aplicable
df.loc[
    df['Engine Type'] == 'Electric',
    'Engine Size Numeric'
] = pd.NA

# Recuperar cilindrada conocida del hibrido 1.5 + Electric
df.loc[
    df['Engine Size (L)'] == '1.5 + Electric',
    'Engine Size Numeric'
] = 1.5

# Recuperar cilindrada conocida de los hibridos de 4.0 L
df.loc[
    df['Engine Size (L)'].isin(['Hybrid (4.0)', '4.0 (Hybrid)']),
    'Engine Size Numeric'
] = 4.0

print(df[['Engine Size (L)', 'Engine Size Numeric', 'Engine Type']].value_counts(dropna=False))

Engine Size (L)       Engine Size Numeric  Engine Type
4                     4.0                  Combustion     219
6.2                   6.2                  Combustion     113
3                     3.0                  Combustion      85
3.5                   3.5                  Combustion      79
5                     5.0                  Combustion      68
6.5                   6.5                  Combustion      46
3.8                   3.8                  Combustion      38
Electric              NaN                  Electric        36
3.7                   3.7                  Combustion      35
2                     2.0                  Combustion      34
3.9                   3.9                  Combustion      30
2.9                   2.9                  Combustion      30
5.2                   5.2                  Combustion      29
6                     6.0                  Combustion      28
2.5                   2.5                  Combustion      25
8              

### Hallazgos — Engine Size

La variable `Engine Size (L)` contiene una combinación de valores numéricos y descripciones como `Electric`, `Electric Motor`, `Hybrid`, `1.5 + Electric` y otras variantes. También se encontraron valores faltantes y registros representados mediante `0` o `-`.

La investigación mostró que los registros `0` y `-` corresponden al Tesla Roadster, por lo que fueron identificados como vehículos eléctricos.

### Tratamiento

Se creó la variable categórica `Engine Type` para clasificar los vehículos como:

- `Combustion`
- `Hybrid`
- `Electric`
- `Unknown`

También se creó `Engine Size Numeric` para conservar exclusivamente la cilindrada expresada en litros. En los vehículos eléctricos se mantuvo este valor como `NaN`, ya que la cilindrada no es aplicable.

Cuando la cilindrada de un vehículo híbrido estaba explícitamente indicada en el valor original, esta fue recuperada y conservada.

### Resultado

Se obtuvo una variable numérica utilizable para la cilindrada y una nueva variable categórica que conserva la información sobre el tipo de motorización sin eliminar información relevante del dataset.

# Trabajando con la columna **HorsePower**

In [10]:
#investigacion preliminar de la columna 

horsepower_prueba = pd.to_numeric(
    df['Horsepower'],
    errors='coerce'
)
print("Cantidad de valores que no pudieron convertirse:")
print(horsepower_prueba.isna().sum())

print("\nValores originales que no pudieron convertirse:")
print(
    df.loc[horsepower_prueba.isna(), 'Horsepower'].value_counts(dropna=False)
)

Cantidad de valores que no pudieron convertirse:
9

Valores originales que no pudieron convertirse:
Horsepower
1000+      3
1,000+     1
10000+     1
10,000     1
1,500      1
10,000+    1
1,020      1
Name: count, dtype: int64


Durante la investigacion preliminar se descubrieron registros con datos sospechosos. Los valores de 1,000 a 1,500 HP son totalmente plausibles en un dataset de autos deportivos. El problema es simplemente que contienen "," o "+". Aqui el "+" significa realmente "1000 o mas", asi que convertirlo exactamente a 1000 implica una simplificacion. Sin embargo tambien hay registros con "10,000+" y eso seria un valor extremo respecto al resto del dataset y podria afetar muchisimo al modelo.

In [11]:
#Crearemos una lista para encacillar los valores sospechosos y poder hacer consulta mas detallada con dichos datos

valores_sospechosos = ['10000+', '10,000', '10,000+']

print(
    df[df['Horsepower'].isin(valores_sospechosos)][
        [

        'Car Make',
        'Car Model',
        'Year',
        'Horsepower',
        'Torque (lb-ft)',
        'Price (in USD)'
        ]
    ]
)

    Car Make Car Model  Year Horsepower Torque (lb-ft) Price (in USD)
389    Tesla  Roadster  2022     10000+              0        200,000
885    Tesla  Roadster  2022     10,000          7,376        200,000
916    Tesla  Roadster  2022    10,000+            NaN        200,000


Durante la exploración especifica datos sospechosos se identificaron tres registros del Tesla Roadster con valores de aproximadamente 10,000 HP. Estos valores fueron considerados sospechosos debido a su magnitud extrema y a inconsistencias adicionales observadas en los mismos registros. Para evitar introducir valores potencialmente erróneos en el modelo, se decidió tratarlos como datos faltantes (NaN) en lugar de modificar su valor sin evidencia suficiente.

In [12]:
#Crear una nueva columna numerica de horsepower.
df['Horsepower Numeric'] = (
    df['Horsepower']
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
)

df['Horsepower Numeric'] = pd.to_numeric(
    df['Horsepower Numeric'],
    errors='coerce'
)

#Los valores de 10,000 HP fueron considerados datos no confiables
df.loc[
    df['Horsepower Numeric'] == 10000,
    'Horsepower Numeric'
] = pd.NA

print('Cantidad de datos No confiables: ', df['Horsepower Numeric'].isna().sum())
print()
print('Estadisticas de la Columna "Horsepower Numeric: ')
print(df['Horsepower Numeric'].describe())

Cantidad de datos No confiables:  3

Estadisticas de la Columna "Horsepower Numeric: 
count    1004.000000
mean      630.069721
std       301.505146
min       181.000000
25%       454.000000
50%       591.000000
75%       690.000000
max      2000.000000
Name: Horsepower Numeric, dtype: float64


### Tratamiento de Horsepower: 

Se creó `Horsepower Numeric`, eliminando las comas y signos `+` antes de convertir los valores a formato numérico.

Los tres registros cercanos a `10,000 HP` fueron considerados no confiables y convertidos a `NaN`, evitando asignar valores corregidos sin evidencia suficiente.


### Resultado

La nueva variable contiene valores numéricos entre 181 y 2,000 HP, mientras que los tres registros considerados no confiables permanecen como valores faltantes para su posterior tratamiento.

# Trabajando con la columna Torque (lb-ft)



In [13]:
#Investigacion preliminar de la columna Torque

torque_prueba = pd.to_numeric(
    df['Torque (lb-ft)'],
    errors="coerce"
)

print("Cantidad de valores que no pudieron convertirse:", torque_prueba.isna().sum())
print("\nValores originales que no pudieron convertirse:")
print(
    df.loc[
        torque_prueba.isna(),
        'Torque (lb-ft)'
    ].value_counts(dropna=False)
)

#revision de distribucion estadistica preliminar
print('\nEstadisticas preliminares')
print(torque_prueba.describe())

Cantidad de valores que no pudieron convertirse: 8

Valores originales que no pudieron convertirse:
Torque (lb-ft)
NaN        3
-          1
10,000+    1
7,376      1
1,180      1
1,050      1
Name: count, dtype: int64

Estadisticas preliminares
count     999.000000
mean      542.185185
std       242.509345
min         0.000000
25%       406.000000
50%       509.000000
75%       604.000000
max      1732.000000
Name: Torque (lb-ft), dtype: float64


In [14]:
#Crearemos una lista para encacillar los valores sospechosos y poder hacer consulta mas detallada con dichos datos

torque_sospechosos = ['10,000+', '7,376']

print( 
    df[df['Torque (lb-ft)'].isin(torque_sospechosos)][

        [
        'Car Make',
        'Car Model',
        'Year',
        'Engine Size (L)',
        'Horsepower',
        'Torque (lb-ft)',
        'Price (in USD)'
        ]
    ]
)

    Car Make Car Model  Year Engine Size (L) Horsepower Torque (lb-ft)  \
354    Tesla  Roadster  2022        Electric      1000+        10,000+   
885    Tesla  Roadster  2022               0     10,000          7,376   

    Price (in USD)  
354        200,000  
885        200,000  


In [15]:
#Crear una nueva columna numerica de Torque

df['Torque Numeric'] = (
    df['Torque (lb-ft)']
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
)

df['Torque Numeric'] = pd.to_numeric(
    df['Torque Numeric'],
    errors="coerce"
)

#Los valors extremos identificados del Tesla Roadster fueron considerados datos no confiable

df.loc[
    df['Torque Numeric'].isin([7376, 10000]),
    'Torque Numeric'
]=pd.NA

#Verificar los resultados de todo el procedimiento

print('Cantidad de valores faltantes/no confiables:')
print(df['Torque Numeric'].isna().sum())
print()
print('Datos estadisticos de finales')
print(df['Torque Numeric'].describe())


Cantidad de valores faltantes/no confiables:
6

Datos estadisticos de finales
count    1001.000000
mean      543.329670
std       243.631963
min         0.000000
25%       406.000000
50%       509.000000
75%       604.000000
max      1732.000000
Name: Torque Numeric, dtype: float64


### Hallazgos — Torque

La variable `Torque (lb-ft)` contiene valores almacenados como texto debido a caracteres como comas y signos `+`. También se identificaron valores faltantes y un registro representado mediante `-`.

Durante la investigación se encontraron valores extremos de `7,376` y `10,000+ lb-ft` asociados al Tesla Roadster. Debido a las inconsistencias presentes en estos registros, fueron considerados no confiables.

### Tratamiento

Se creó `Torque Numeric`, eliminando los caracteres que impedían la conversión y transformando los valores válidos a formato numérico.

Los valores `7,376` y `10,000+ lb-ft` fueron convertidos a `NaN`. Otros valores altos, como `1,180` y `1,050 lb-ft`, fueron conservados al no encontrarse evidencia suficiente para considerarlos errores.

### Resultado

La variable quedó disponible en formato numérico y los seis registros faltantes o no confiables se conservaron como `NaN` para su posterior tratamiento durante la preparación de los datos.

# Trabajando con la columna '0-60 MPH Time (seconds)'

In [16]:
#investigacion preliminar de la columna

tiempo_prueba = pd.to_numeric(
    df['0-60 MPH Time (seconds)'],
    errors="coerce"
)

print("Cantidad de valores que no pudieron convertirse")
print(tiempo_prueba.isna().sum())

print("\nValores originales que no pudieron convertirse")
print(
    df.loc[
        tiempo_prueba.isna(),
        '0-60 MPH Time (seconds)'
        ].value_counts(dropna=False)
)

print('\nEstadisticas Preliminares')
print(tiempo_prueba.describe())


Cantidad de valores que no pudieron convertirse
1

Valores originales que no pudieron convertirse
0-60 MPH Time (seconds)
< 1.9    1
Name: count, dtype: int64

Estadisticas Preliminares
count    1006.000000
mean        3.515010
std         0.776358
min         1.800000
25%         2.900000
50%         3.500000
75%         4.000000
max         6.500000
Name: 0-60 MPH Time (seconds), dtype: float64


In [17]:
#investigar que vehiculo es el del dato sospechoso

print(
    df[df['0-60 MPH Time (seconds)'] == '< 1.9'][

        [
            'Car Make',
            'Car Model',
            'Year',
            'Horsepower',
            'Torque (lb-ft)',
            '0-60 MPH Time (seconds)',
            'Price (in USD)'


        ]
    ]
)

    Car Make Car Model  Year Horsepower Torque (lb-ft)  \
364    Tesla  Roadster  2023     1,000+            737   

    0-60 MPH Time (seconds) Price (in USD)  
364                   < 1.9        200,000  


In [18]:
#crear columna numerica para el tiempo de aceleracion

df['0-60 MPH Time Numeric'] = (
    df['0-60 MPH Time (seconds)']
    .str.replace('<','', regex=False)
    .str.strip()
)

df['0-60 MPH Time Numeric'] = pd.to_numeric(

    df['0-60 MPH Time Numeric'],
    errors="coerce"
)

print('Cantidad de valores faltantes:')
print(df['0-60 MPH Time Numeric'].isna().sum())

print('\nEstadisticas')
print(df['0-60 MPH Time Numeric'].describe())

Cantidad de valores faltantes:
0

Estadisticas
count    1007.000000
mean        3.513406
std         0.777639
min         1.800000
25%         2.900000
50%         3.500000
75%         4.000000
max         6.500000
Name: 0-60 MPH Time Numeric, dtype: float64


### Hallazgos — 0-60 MPH Time

De los 1,007 registros, 1,006 podían convertirse directamente a formato numérico. El único valor incompatible fue `< 1.9`, correspondiente a un Tesla Roadster.

El dato contiene información válida; el problema se debe únicamente al símbolo `<`, que impide su conversión directa.

### Tratamiento

Se creó `0-60 MPH Time Numeric`, eliminando el símbolo `<` y los espacios adicionales.

El valor `< 1.9` fue representado como `1.9` segundos para permitir su utilización en el modelo, reconociendo que constituye una aproximación del valor original.

### Resultado

Los 1,007 registros pudieron representarse numéricamente y la nueva variable no contiene valores faltantes.

# Trabajando con la Columna Target - Price (in USD)

In [19]:
#investigacion preliminar de la columna

price_prueba = pd.to_numeric(
    df['Price (in USD)'],
    errors="coerce"
)

print("Cantidad de valores que no pudieron convertirse", price_prueba.isna().sum())
print('\nValores originales que no pudieron convertirse')
print( 
    df.loc[
        price_prueba.isna(),
        'Price (in USD)'
    ].value_counts(dropna=False)
    
)

print('\nEstadisticas Preliminares de "Price (in USD)" ')
print(price_prueba.describe())

Cantidad de valores que no pudieron convertirse 1007

Valores originales que no pudieron convertirse
Price (in USD)
500,000      34
3,000,000    24
625,000      22
58,900       17
92,950       16
             ..
78,100        1
78,450        1
3,900,000     1
254,500       1
27,205        1
Name: count, Length: 367, dtype: int64

Estadisticas Preliminares de "Price (in USD)" 
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: Price (in USD), dtype: float64


In [20]:
#Busqueda de valores que contengan caracteres diferentes de numeros y comas

valores_raros_price = df[
    df['Price (in USD)'].str.contains(
        r'[^0-9,]',
        regex=True,
        na=False
    )
]['Price (in USD)']

print("Cantidad de valores con caracteres especiales:", len(valores_raros_price))

print("\nValores encontrados:")
print(valores_raros_price.value_counts(dropna=False))

#Busqueda de valores faltantes en la variable original 'Price (in USD)'ArithmeticError

print("\nValores NaN originales:", df['Price (in USD)'].isna().sum())

Cantidad de valores con caracteres especiales: 0

Valores encontrados:
Series([], Name: count, dtype: int64)

Valores NaN originales: 0


In [21]:
#Limpieza temporal en otra variable de prueba

price_limpio_prueba = pd.to_numeric(
    df['Price (in USD)'].str.replace(',', '', regex=False),
    errors='coerce'
    )

print("Valores que siguen sin convertirse:", price_limpio_prueba.isna().sum())
print("\nEstadisticas Preliminares")
print(price_limpio_prueba.describe().apply(lambda x: f'{x:,.2f}'))


Valores que siguen sin convertirse: 0

Estadisticas Preliminares
count        1,007.00
mean       382,035.94
std        738,322.72
min         25,000.00
25%         71,800.00
50%        140,000.00
75%        250,000.00
max      5,200,000.00
Name: Price (in USD), dtype: str


In [22]:
#crear una copia temporal para investigar los precios 

df_price_prueba = df.copy()

df_price_prueba['Price Prueba'] = price_limpio_prueba

print("Los 15 vehiculos con precios mas altos:")
print(
    df_price_prueba.sort_values('Price Prueba', ascending=False)[
        [
            'Car Make',
            'Car Model',
            'Year',
            'Horsepower',
            'Torque (lb-ft)',
            '0-60 MPH Time (seconds)',
            'Price (in USD)',
            'Price Prueba'
        ]
    ].head(15)
)

Los 15 vehiculos con precios mas altos:
         Car Make                Car Model  Year Horsepower Torque (lb-ft)  \
823       Bugatti  Chiron Super Sport 300+  2021       1578           1180   
541       Bugatti  Chiron Super Sport 300+  2022       1578           1180   
983       Bugatti                   Chiron  2022       1500           1180   
438   Lamborghini                     Sián  2021        819            531   
624       Bugatti         Chiron Pur Sport  2021       1500           1180   
279        Pagani       Huayra Roadster BC  2021        791            774   
385        Pagani                   Huayra  2021        764            738   
174      W Motors         Lykan Hypersport  2015        780            708   
328    Koenigsegg                    Jesko  2022       1280           1015   
376       Bugatti                   Chiron  2022       1500           1180   
1002   Koenigsegg                    Jesko  2022       1280           1106   
1001      Bugatti       

In [23]:
print("\nLos 10 vehiculos con precios mas bajos:")
print(
    df_price_prueba
    .sort_values('Price Prueba')[
        [
            'Car Make',
            'Car Model',
            'Year',
            'Horsepower',
            'Price (in USD)',
            'Price Prueba'
        ]
    ]
    .head(10)
)


Los 10 vehiculos con precios mas bajos:
      Car Make   Car Model  Year Horsepower Price (in USD)  Price Prueba
997  Chevrolet      Camaro  2021        455         25,000         25000
597  Chevrolet      Camaro  2021        455         25,000         25000
924  Chevrolet      Camaro  2021        455         25,000         25000
886  Chevrolet      Camaro  2022        455         25,000         25000
92       Mazda  MX-5 Miata  2021        181         26,830         26830
998       Ford     Mustang  2021        310         27,205         27205
707      Dodge  Challenger  2022        305         28,000         28000
317     Nissan        370Z  2021        332         30,000         30000
895     Nissan        370Z  2021        332         30,000         30000
779     Nissan  370Z Coupe  2020        332         30,090         30090


In [24]:
#Crear Columna numerica para price

df['Price Numeric'] = (
    df['Price (in USD)']
    .str.replace(',', '', regex=False)
)

df['Price Numeric'] = pd.to_numeric(
    df['Price Numeric'],
    errors="coerce"
)

print("Cantidad de valores faltantes: ", df['Price Numeric'].isna().sum())
print('\nEstadisticas de Price Numeric:')
print(df['Price Numeric'].describe().apply(lambda x: f'{x:,.2f}'))

Cantidad de valores faltantes:  0

Estadisticas de Price Numeric:
count        1,007.00
mean       382,035.94
std        738,322.72
min         25,000.00
25%         71,800.00
50%        140,000.00
75%        250,000.00
max      5,200,000.00
Name: Price Numeric, dtype: str


### Hallazgos — Price

Los 1,007 registros de `Price (in USD)` estaban almacenados como texto debido al uso de comas como separadores de miles.

No se encontraron valores faltantes ni otros caracteres especiales. Después de eliminar temporalmente las comas, todos los registros pudieron convertirse correctamente a formato numérico.

La distribución presenta una diferencia considerable entre la media y la mediana debido a la presencia de vehículos de muy alto valor. La revisión de los extremos mostró que estos precios corresponden principalmente a modelos de marcas como Bugatti, Pagani, Koenigsegg y Lamborghini, por lo que no se consideraron errores.

### Tratamiento

Se creó `Price Numeric`, eliminando las comas y convirtiendo los valores a formato numérico.

Los valores extremos fueron conservados debido a que representan segmentos reales dentro del dataset y no existe evidencia suficiente para eliminarlos o modificarlos.

### Resultado

Los 1,007 registros fueron convertidos correctamente y no se generaron valores faltantes. `Price Numeric` será utilizada como variable objetivo (`target`) del modelo de regresión.

In [25]:
#Resultados luego de los cambios


print("Dimensiones nuevas del dataset:", df.shape)
print("Columnas del datset luego de los cambios:")
print(list(df.columns))
print("\nTipo de datos del dataset:")
print(df.dtypes)
print("\nValores nulos en el dataset:")
print(df.isnull().sum())

Dimensiones nuevas del dataset: (1007, 14)
Columnas del datset luego de los cambios:
['Car Make', 'Car Model', 'Year', 'Engine Size (L)', 'Horsepower', 'Torque (lb-ft)', '0-60 MPH Time (seconds)', 'Price (in USD)', 'Engine Type', 'Engine Size Numeric', 'Horsepower Numeric', 'Torque Numeric', '0-60 MPH Time Numeric', 'Price Numeric']

Tipo de datos del dataset:
Car Make                       str
Car Model                      str
Year                         int64
Engine Size (L)                str
Horsepower                     str
Torque (lb-ft)                 str
0-60 MPH Time (seconds)        str
Price (in USD)                 str
Engine Type                    str
Engine Size Numeric        float64
Horsepower Numeric         float64
Torque Numeric             float64
0-60 MPH Time Numeric      float64
Price Numeric                int64
dtype: object

Valores nulos en el dataset:
Car Make                    0
Car Model                   0
Year                        0
Engine Size (

### Trabajando con los Valores Nulos

In [26]:
#De los valores nulos en el print anterior buscaremos dichos nulos de cada variable antes de imputar o rellenarlos.
print("NaN de Engine Size Numeric por tipo de motor:")
print(df[df['Engine Size Numeric'].isna()]['Engine Type'].value_counts(dropna=False))

print("\nRegistros con Horsepower Numeric Faltante:")
print(
    df[df['Horsepower Numeric'].isna()][
        [
            'Car Make',
            'Car Model',
            'Year',
            'Engine Type',
            'Horsepower',
            'Horsepower Numeric'

        ]

    ]
)

print("\nRegistros con Torque Numeric faltantes:")
print(
    df[df['Torque Numeric'].isna()][
        [
            'Car Make',
            'Car Model',
            'Year',
            'Engine Type',
            'Torque (lb-ft)',
            'Torque Numeric'
        ]
    ]
)

NaN de Engine Size Numeric por tipo de motor:
Engine Type
Electric    45
Unknown     10
Hybrid       2
Name: count, dtype: int64

Registros con Horsepower Numeric Faltante:
    Car Make Car Model  Year Engine Type Horsepower  Horsepower Numeric
389    Tesla  Roadster  2022     Unknown     10000+                 NaN
885    Tesla  Roadster  2022    Electric     10,000                 NaN
916    Tesla  Roadster  2022     Unknown    10,000+                 NaN

Registros con Torque Numeric faltantes:
     Car Make      Car Model  Year Engine Type Torque (lb-ft)  Torque Numeric
335     Tesla       Roadster  2022    Electric              -             NaN
354     Tesla       Roadster  2022    Electric        10,000+             NaN
642     Tesla  Model S Plaid  2021    Electric            NaN             NaN
878  Maserati    GranTurismo  2021    Electric            NaN             NaN
885     Tesla       Roadster  2022    Electric          7,376             NaN
916     Tesla       Roadster  

## Análisis final de valores faltantes

Después de completar las transformaciones se identificaron valores `NaN` en tres variables:

- **Engine Size Numeric:** 57 valores faltantes. La mayoría corresponden a vehículos eléctricos, donde la cilindrada en litros no aplica. Los restantes pertenecen a vehículos híbridos sin cilindrada especificada o registros clasificados como `Unknown`.

- **Horsepower Numeric:** 3 valores faltantes correspondientes a registros del Tesla Roadster con valores cercanos a `10,000 HP`, previamente identificados como no confiables.

- **Torque Numeric:** 6 valores faltantes provenientes tanto de datos originalmente ausentes como de valores extremos considerados no confiables.

Los valores faltantes no serán imputados durante esta fase. Su tratamiento se realizará durante la preparación de los datos para Machine Learning y después del `train/test split`, permitiendo calcular cualquier estadística de imputación exclusivamente con los datos de entrenamiento y evitando **data leakage**.

Las variables `0-60 MPH Time Numeric` y `Price Numeric` quedaron completas y no presentan valores faltantes.

In [ ]:
print(df.isnull().sum())

In [27]:
#Guardar El dataset listo para la preparacion de datos de ML antes de usarlos en el modelo.
#print(os.getcwd()) Nos da la ruta actual de donde esta nuestro notebook
#print(os.path.exists('../datasets')) Este codigo nos devuelve verdadero si en dicha ruta establecida existe un directorio o no
df.to_csv('../datasets/Sport-Car-Price-Clean.csv', index=False)